# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {'A': 2, 'B': 1}. r.get(c, 0) возвращает накопленный счётчик по ключу или ноль, а цикл дважды встречает A.
2. [0, 1, 1]. В функцию передаётся ссылка на тот же изменяемый список; каждый вызов append(1) добавляет ещё один элемент.
3. 2.Приведение к нижнему регистру делает Almaty и ALMATY одинаковыми, а множество хранит только уникальные значения.
4. {'ab': 2, 'abc': 3}. У словаря ключи уникальны: повторный ключ ab перезаписывается значением той же длины.
5. Сначала 5, затем None. g(5) печатает 5, но не содержит return, поэтому возвращает None, который затем выводится print(x).

In [1]:
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [5]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [8]:
def to_num(s):
    return float(s.strip().replace(" ", "").replace(",", "."))
def clean_city(s):
    key = s.strip().lower()
    latin_to_russian = {
        "almaty": "Алматы",
        "astana": "Астана",
        "shymkent": "Шымкент",
    }
    return latin_to_russian.get(key, key.title())
print(to_num("15 990"), clean_city(" Astana "))
category_counts = {}
for row in rows:
    category = row["category"]
    category_counts[category] = category_counts.get(category, 0) + 1
for category, count in sorted(category_counts.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{category:<12} | {count:>5}")
category_revenue = {}
category_paid_count = {}
for row in rows:
    if row["status"] == "оплачен":
        category = row["category"]
        category_revenue[category] = category_revenue.get(category, 0) + to_num(row["amount"])
        category_paid_count[category] = category_paid_count.get(category, 0) + 1
print("\nКатегория    | Оплачено, ₸ | Оплачено заказов | Средний чек, ₸")
for category in sorted(category_revenue):
    revenue = category_revenue[category]
    count = category_paid_count[category]
    average = revenue / count
    print(f"{category:<12} | {revenue:>12,.0f} | {count:>14} | {average:>15,.0f}")
monthly_counts = {}
for row in rows:
    month = row["order_date"][:7]
    monthly_counts[month] = monthly_counts.get(month, 0) + 1
for month in sorted(monthly_counts):
    print(f"{month}: {monthly_counts[month]}")

15990.0 Астана
Одежда       |  1529
Электроника  |  1185
Дом          |  1105
Красота      |   991
Книги        |   653

Категория    | Оплачено, ₸ | Оплачено заказов | Средний чек, ₸
Дом          |   10,516,320 |            987 |          10,655
Книги        |    2,294,100 |            583 |           3,935
Красота      |    5,507,620 |            890 |           6,188
Одежда       |   18,321,080 |           1379 |          13,286
Электроника  |   65,270,240 |           1065 |          61,287
2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [6]:
# Стандарт

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [7]:
# Челлендж